# AI-Powered Financial Fraud Detection and Risk Analysis System

# Notebook 5: Model Evaluation & Fraud Risk Intelligence

## Objective

Evaluate the Isolation Forest and AutoEncoder models, compare their performance, generate fraud risk intelligence, create actionable recommendations, and export dashboard-ready datasets.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

pd.set_option("display.max_columns",None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Prediction Files

In [2]:
iso = pd.read_csv("../data/processed/isolation_forest_predictions.csv")
ae = pd.read_csv("../data/processed/autoencoder_predictions.csv")

print("Isolation Forest:",iso.shape)
print("AutoEncoder:",ae.shape)

Isolation Forest: (283726, 38)
AutoEncoder: (283726, 38)


## Model Evaluation Function

In [3]:
def evaluate(df,name):
    y=df["Class"]
    p=df["Prediction"]

    return {
        "Model":name,
        "Accuracy":accuracy_score(y,p),
        "Precision":precision_score(y,p),
        "Recall":recall_score(y,p),
        "F1 Score":f1_score(y,p)
    }

results=pd.DataFrame([
    evaluate(iso,"Isolation Forest"),
    evaluate(ae,"AutoEncoder")
])

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Isolation Forest,0.997043,0.177817,0.213531,0.194044
1,AutoEncoder,0.989694,0.118818,0.807611,0.207158


## Compare Models

In [4]:
fig=px.bar(
    results,
    x="Model",
    y=["Accuracy","Precision","Recall","F1 Score"],
    barmode="group",
    title="Model Performance Comparison"
)
fig.show()

## Best Model

In [5]:
best_model=results.sort_values("F1 Score",ascending=False).iloc[0]["Model"]
print("Best Model:",best_model)

Best Model: AutoEncoder


## Fraud Risk Intelligence

In [6]:
df = ae.copy() if best_model=="AutoEncoder" else iso.copy()

if "Recommendation" not in df.columns:
    def recommendation(level):
        if level=="High":
            return "Block Transaction"
        elif level=="Medium":
            return "Manual Verification"
        return "Approve"
    df["Recommendation"]=df["Risk_Level"].apply(recommendation)

priority_map={"Low":"P3","Medium":"P2","High":"P1"}
df["Alert_Priority"]=df["Risk_Level"].map(priority_map)

df["Alert"]=np.where(df["Prediction"]==1,"Fraud Alert","Safe")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,Scaled_Amount,Hour,Time_Period,Prediction,Reconstruction_Error,Risk_Score,Risk_Level,Recommendation,Alert_Priority,Alert
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0,0.244200,0,Night,0,0.155549,0.22,Low,Approve,P3,Safe
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0,-0.342584,0,Night,0,0.131470,0.19,Low,Approve,P3,Safe
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,1.158900,0,Night,0,0.170155,0.24,Low,Approve,P3,Safe
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0,0.139886,0,Night,0,0.311243,0.45,Low,Approve,P3,Safe
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0,-0.073813,0,Night,0,0.253162,0.36,Low,Approve,P3,Safe


## Fraud Alerts

In [7]:
alerts=df[df["Prediction"]==1].sort_values("Risk_Score",ascending=False)

alerts[[
    "Amount",
    "Risk_Score",
    "Risk_Level",
    "Recommendation",
    "Alert_Priority"
]].head(20)

,Amount,Risk_Score,Risk_Level,Recommendation,Alert_Priority
154090,0.01,100.00,High,Block Transaction,P1
153993,0.00,99.01,High,Block Transaction,P1
153640,1.00,98.53,High,Block Transaction,P1
153777,2.28,97.80,High,Block Transaction,P1
150418,1.00,94.60,High,Block Transaction,P1
153242,1.00,91.91,High,Block Transaction,P1
153230,2.28,87.43,High,Block Transaction,P1
151704,9.82,85.71,High,Block Transaction,P1
44087,1.00,82.88,High,Block Transaction,P1
44040,139.90,82.84,High,Block Transaction,P1


## Risk Level Distribution

In [8]:
fig=px.pie(
    df,
    names="Risk_Level",
    title="Risk Level Distribution"
)
fig.show()

## Dashboard KPIs

In [9]:
kpis=pd.DataFrame({
"Metric":[
"Total Transactions",
"Fraud Alerts",
"Safe Transactions",
"High Risk",
"Medium Risk",
"Low Risk"],
"Value":[
len(df),
(df["Prediction"]==1).sum(),
(df["Prediction"]==0).sum(),
(df["Risk_Level"]=="High").sum(),
(df["Risk_Level"]=="Medium").sum(),
(df["Risk_Level"]=="Low").sum()
]})
kpis

,Metric,Value
0,Total Transactions,283726
1,Fraud Alerts,3215
2,Safe Transactions,280511
3,High Risk,55
4,Medium Risk,83
5,Low Risk,283588


## Save Dashboard Files

In [10]:
results.to_csv("../reports/model_comparison.csv",index=False)
alerts.to_csv("../reports/fraud_alerts.csv",index=False)
kpis.to_csv("../reports/dashboard_kpis.csv",index=False)

df.to_csv("../data/processed/final_dashboard_dataset.csv",index=False)

print("Dashboard datasets exported successfully.")

Dashboard datasets exported successfully.


# Business Insights

- Both anomaly detection models were evaluated on the same processed dataset.
- The model with the strongest F1-score is selected for deployment.
- Each suspicious transaction is enriched with a risk level, recommendation, and alert priority.
- The exported datasets are ready for direct use in the Streamlit dashboard.


# Conclusion

This notebook completes the fraud intelligence pipeline by converting model outputs into actionable business information.

**Next Notebook:** Dashboard Integration & Model Loading.
